# מפגש 21 — מחברת מודרכת Audited
**90 דקות הוראה ותרגול + 45 דקות פרויקט**. דודו למד שאלות בעל פה; איך נדע מה יקרה בשאלה שלא ראה? נתוני A/B/C מומצאים וממחישים תוצאות מודלים, **לא אימון מודל שהתבצע במחברת**.

בסיום: תזהו סימן אפשרי להתאמת יתר/חסר; תבחינו Train, Validation, Test; ותדעו שלא בוחרים לפי Test. אין כאן קוד Cross Validation (במפגש 22).

## 1. שאלת חיזוי לפני הקוד
A משיג 1.00 באימון ו־0.65 באימות. כמה נקודות אחוז מפרידות ביניהם? האם 100% באימון מוכיח הצלחה בעתיד?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
data=pd.DataFrame({"candidate":["A","B","C"],"train":[1.00,0.88,0.60],"validation":[0.65,0.84,0.58]})
assert data.shape==(3,3)
assert list(data["candidate"])==["A","B","C"]
assert data["train"].between(0,1).all()
assert data["validation"].between(0,1).all()
assert data["candidate"].is_unique
assert data["train"].notna().all()
assert data["validation"].notna().all()
display(data)

## 2. הגדרת מדד הפער
Gap = Train score − Validation score. הפער נמדד באותן יחידות כמו המדד; מעבר מ־1.00 ל־0.65 הוא 35 **נקודות אחוז**, לא ירידה יחסית של 35% מערך האימון. אין לשנות את ציוני המקור.

In [ ]:
manual_A=1.00-0.65
assert np.isclose(manual_A,0.35)
assert np.isclose(manual_A*100,35)
data["gap"]=data["train"]-data["validation"]
assert len(data["gap"])==3
assert np.allclose(data["gap"],[.35,.04,.02])
assert np.isclose(data.loc[0,"gap"],manual_A)
assert data["gap"].between(0,1).all()
print("Gap A:",manual_A)

## 3. גרף: רמת ביצועים מול פער
השוו גובה העמודות בנוסף למרחק ביניהן. פער קטן כשגם האימון וגם האימות חלשים אינו הוכחה להכללה טובה. הטבלה עצמה אינה מוכיחה את *סיבת* הפער.

In [ ]:
ax=data.plot(x="candidate",y=["train","validation"],kind="bar",rot=0,ylim=(0,1.05),title="Synthetic train and validation scores")
ax.set_ylabel("Score (0–1)")
plt.show()
assert data["gap"].max()<=1
assert data["gap"].min()>=0

## 4. מהי החלטת הבחירה?
לצורך ההסבר בלבד: מדד גדול יותר הוא טוב יותר; קובעים מראש שהבחירה תתבסס על Validation, ולא על Train. תרגיל הבחירה העצמאי נשאר במחברת התלמיד.

In [ ]:
train_leader=data.loc[data["train"].idxmax(),"candidate"]
validation_leader=data.loc[data["validation"].idxmax(),"candidate"]
assert train_leader in data["candidate"].to_numpy()
assert validation_leader in data["candidate"].to_numpy()
assert 0<=data["validation"].max()<=1
assert len(data["candidate"].unique())==len(data)
print("Training leader differs from validation leader?",train_leader!=validation_leader)

## 5. חלוקת תפקידים — בלי תוצאות Test
Train: לומדים; Validation: משווים ובוחרים; Test: מודדים בסוף. אלו שלוש מטרות נבדלות. החזקת Test בצד אינה אומרת שניתן לבחור מחדש לאחר שצפינו בו.

In [ ]:
roles={"Train":"fit","Validation":"choose","Test":"final"}
assert len(roles)==3
assert roles["Train"]=="fit"
assert roles["Validation"]=="choose"
assert roles["Test"]=="final"
assert len(set(roles.values()))==3
print(roles)

## 6. המחשת חלוקה סכמטית (לא ניסוי מודל)
רק כדי לראות הפרדה בין קבוצות, יוצרים 20 מספרי מזהה מלאכותיים ומחלקים 15/5. **לא** מחשבים ציוני אמת ואין אימון מודל בתא זה.

In [ ]:
from sklearn.model_selection import train_test_split
ids=np.arange(20)
train_ids,test_ids=train_test_split(ids,test_size=0.25,random_state=42)
assert len(train_ids)==15
assert len(test_ids)==5
assert set(train_ids).isdisjoint(test_ids)
assert set(train_ids).union(test_ids)==set(ids)
assert len(set(ids))==20
print("Train count:",len(train_ids),"Test count:",len(test_ids))

## 7. קטע שגוי לדיון — לא להרצה
'ראיתי שתוצאת Test נמוכה, לכן החלפתי את המועמד והגרלתי seed אחר'. איזו טעות מחקרית קרתה? מדוע החלפת seed אינה מוחקת חשיפה שכבר השפיעה על ההחלטה?

In [ ]:
protocol=["fix metric","fix candidate list","compare on validation","lock choice","evaluate on test"]
assert len(protocol)==5
assert protocol.index("lock choice")<protocol.index("evaluate on test")
assert protocol.index("compare on validation")<protocol.index("lock choice")
assert "choose using test" not in protocol
print("Protocol:", " -> ".join(protocol))

## כרטיס יציאה
הסבירו במילים התאמת חסר, התאמת יתר והכללה. ספרו מדוע אסור לקבוע סיבתיות מפער אחד בלבד, ומה היה קורה אם תקנון היה מותאם על נתוני האימות. שמרו 45 דקות לתכנון ניסוי הפרויקט.